# 1. 认识进程

## 1.1 什么是进程？

进程是“正在运行的程序”，比如打开微信、浏览器、Python 脚本，都会创建一个进程。

操作系统会给每个进程分配独立的内存、CPU 等资源，进程之间互不干扰 —— 就像多个独立的房子，每个房子有自己的家具（资源），房子之间互不影响。

## 1.2 进程 vs 线程（核心区别）

| 对比维度 | 进程 | 线程 |
|----------|------|------|
| 资源分配 | 独立资源空间（内存、CPU），资源占用多 | 共享进程资源，资源占用少 |
| 通信难度 | 需要专门通信机制（如队列），难度高 | 共享全局变量，通信方便 |
| 执行效率 | 创建 / 切换成本高，效率低 | 创建 / 切换成本低，效率高 |
| 多核利用 | 支持真正并行，充分利用多核 CPU | 受 GIL 限制，多核利用率低（Python） |
| 稳定性 | 一个进程崩溃不影响其他进程，稳定性高 | 一个线程崩溃可能导致整个进程崩溃 |
| 适用场景 | CPU 密集型任务（大量计算） | I/O 密集型任务（文件读写、网络请求） |

## 1.3 进程的 5 种状态（了解即可）

进程在运行过程中会经历多种状态，由操作系统调度：

1. **新建态**：进程刚被创建，但尚未完成初始化，未进入就绪队列（比如双击软件启动）；
2. **就绪态**：具备执行条件，等待 CPU 调度（比如打开多个应用程序，但 CPU 同一时间只能执行一个进程，其他进程在就绪队列中排队等待）；
3. **运行态**：CPU 正在执行进程的功能（比如视频播放器正在解码视频）；
4. **阻塞态**：等待某个外部事件发生（比如软件等待用户输入密码）；
5. **终止态**：进程执行完毕或被强制关闭（比如关闭软件）。

# 2. 进程基础：创建与启动进程

Python 通过 `multiprocessing` 模块的 `Process` 类创建进程，步骤和线程类似，但要注意进程的独立性。

## 2.1 基础用法

In [2]:
import time
import multiprocessing

# 定义任务函数
def sing():
    print("正在唱歌...")
    time.sleep(2)
    print("唱歌结束了！")

def dance():
    print("正在跳舞...")
    time.sleep(2)
    print("跳舞结束了！")

if __name__ == "__main__":
    # 创建进程对象
    p1 = multiprocessing.Process(target=sing)
    p2 = multiprocessing.Process(target=dance)

    # 启动进程
    p1.start()
    p2.start()

    # 等待进程结束
    p1.join()
    p2.join()

    print("所有任务完成了！")

所有任务完成了！


## 2.2 进阶用法：给进程传参 + 进程属性

进程传参和线程一样，可以使用 `args`（元组）或 `kwargs`（字典）；

还可以通过属性查看进程信息（如进程 ID、名称）。

In [3]:
def sing(name,age):
    print(f"{name}正在唱歌，年龄是{age}...")
    time.sleep(2)
    print(f"唱歌进程ID{os.getpid()},父进程ID{os.getppid()}，唱歌结束了！")

def dance(name):
    print(f"{name}正在跳舞...")
    time.sleep(2)
    print(f"跳舞进程ID{os.getpid()},父进程ID{os.getppid()}，跳舞结束了！")

if __name__ == "__main__":
    # 创建进程对象
    p1 = multiprocessing.Process(target=sing, args=("小明", 20))
    p2 = multiprocessing.Process(target=dance, args=("小红",))

    # 启动进程
    p1.start()
    p2.start()

    # 等待进程结束
    p1.join()
    p2.join()

    print("所有任务完成了！")

所有任务完成了！


## 2.3 关键特性:进程间不共享全局资源
和线程不同，进程有独立的资源空间，全局变量不会被多个进程共享--每个进程会复制一份全局变量，修改后互不影响。

---

## 3. 进程间通信：Queue 队列（核心方案）

进程间不共享资源，需要通过专门机制通信，`multiprocessing.Queue`（队列）是最常用的方式 —— 先进先出（FIFO），支持多进程安全传递数据。

### 3.1 Queue 队列的基本用法

队列就像“共享信箱”，一个进程往信箱里放数据（入队），另一个进程从信箱里取数据（出队）。  
### 核心方法

- `put(item)`：将数据入队（存数据），如果队列满了，会阻塞等待；
- `get()`：将数据出队（取数据），如果队列为空，会阻塞等待；
- `empty()`：判断队列是否为空（返回 True / False）；
- `full()`：判断队列是否已满（返回 True / False）；
- `qsize()`：获取队列中数据的个数。

### 3.2 实战:进程通过队列通信
用队列实现“生产进程”(写数据)和“消费进程”(读数据)的通信。

In [4]:

import multiprocessing

def producer(q):
    for i in range(5):
        item = f"item-{i}"
        q.put(item)
        print(f"生产了{item}")
        time.sleep(1) 

def consumer(q):
    for i in range(5):
        item = q.get()
        print(f"消费了{item}")
        time.sleep(1)

if __name__ == "__main__":
    q = multiprocessing.Queue()
    
    p1 = multiprocessing.Process(target=producer, args=(q,))
    p2 = multiprocessing.Process(target=consumer, args=(q,))

    p1.start()
    p2.start()

    p1.join()
    p2.join()

    print("生产和消费完成了！")

生产和消费完成了！


## 4. 进程池：高效管理多个进程

如果需要创建几十个、上百个进程，逐个创建会很繁琐，还会浪费资源。  
进程池能预先创建固定数量的进程，重复利用，高效处理大量任务。

### 4.1 什么是进程池？

进程池就像“工厂的生产线”：

预先开 3 条生产线（进程），当有 10 个任务时，3 条生产线同时处理，  
完成一个任务后接着处理下一个，不用为每个任务新建生产线，从而节省资源。

### 4.2 进程池的核心方法

- `Pool(n)`：创建进程池，n 为最大进程数（同时执行的任务数）；
- `apply_async(func, args)`：异步提交任务（主进程不等待，效率高）；
- `close()`：关闭进程池，不再接受新任务；
- `join()`：阻塞主进程，等待进程池所有任务完成（必须在 `close()` 后调用）；
- `get()`：获取异步任务的返回结果。

示例：用进程池处理批量任务

In [ ]:
# 需求:用进程池处理7个计算任务，每个任务耗时约2秒，进程池大小设为3(模拟3核cpu)
# 任务函数:计算数字的3倍，返回结果(模拟cpu密集型任务)
from multiprocessing import Pool

def calculate(num):
    print(f"计算{num}的3倍...")
    time.sleep(2)
    result = num * 3
    print(f"{num}的3倍是{result}")
    return result

if __name__ == "__main__":
    P = Pool(processes=3)  # 创建进程池，大小为3
    result = []  # 用于保存结果对象
    for i in range(7):
        res =P.apply_async(calculate, args=(i,))  # 异步提交任务
        result.append(res)  # 保存结果对象
    P.close()  # 关闭进程池，不再接受新任务
    P.join()  # 等待所有任务完成    
    for res in result:
        print(f"结果: {res.get()}")  # 获取结果并打印

计算0的3倍...
计算1的3倍...
计算2的3倍...
0的3倍是0
计算3的3倍...
2的3倍是6
计算4的3倍...
1的3倍是3
计算5的3倍...
3的3倍是9
计算6的3倍...
4的3倍是12
5的3倍是15
6的3倍是18
结果: 0
结果: 3
结果: 6
结果: 9
结果: 12
结果: 15
结果: 18


# 5. 核心总结
1. 进程核心：独立资源空间，支持多核并行，适合 CPU 密集型任务。

2. 进程创建：multiprocessing.Process(target=任务, args=元组参数)，启动用 start()。

3. 关键特性：
   - 不共享全局资源，需通过 Queue 等机制通信；
   - 进程属性：name（名称）、pid（进程 ID）、is_alive()（是否存活）。

4. 进程通信：Queue 队列是首选，put() 入队、get() 出队，线程 / 进程安全。

5. 进程池：
   - 适合大量任务，Pool(n) 创建，最大进程数 n 建议等于 CPU 核心数；
   - 异步提交用 apply_async()，关闭用 close()，等待完成用 join()。

6. 选型技巧：
   - I/O 密集型（文件读写、网络请求）→ 用线程；
   - CPU 密集型（大量计算）→ 用进程 / 进程池。